# Parte 3: Predicados y escenarios posibles

En esta parte modelarás la clasificación de la fase de grupos del Mundial con **escenarios posibles** y **entailment** sobre mundos completos.

A diferencia de la lógica proposicional, aquí trabajas con un motor genérico (`ScenarioEngine`) que enumera todas las formas de completar partidos pendientes. Tú implementas predicados de ranking y de clasificación/eliminación en `src/worldcup/rules.py`.

## Objetivos
1. Calcular puntos y diferencia de goles sobre un mundo completo (`World`)
2. Comparar equipos en el ranking `(puntos, DG)` con `teams_below` / `teams_above`
3. Entender `PartialState`, mundos posibles y el conteo `3^k`
4. Usar `holds_in_all` para decidir si un equipo está **matemáticamente** clasificado o eliminado
5. Relacionar hechos conocidos vs. conclusiones derivadas (entailment ∀ mundos)


In [ ]:
import sys
sys.path.insert(0, '..')

from src.logic.predicate.engine import (
    Fixture,
    Outcome,
    PartialState,
    PlayedMatch,
    ScenarioEngine,
    World,
)
from src.worldcup.matches import load_matches, state_for

## Paso 0: Familiarízate con el motor

Antes de implementar, explora las clases de `src/logic/predicate/engine.py` (solo lectura):

- `PlayedMatch(home, away, home_goals, away_goals)` — partido con marcador
- `Outcome` — `HOME_WIN` / `DRAW` / `AWAY_WIN` (desde la perspectiva del local)
- `World` — todos los partidos del grupo ya resueltos; métodos útiles: `matches_involving`, `group_rivals`
- `PartialState` — resultados **conocidos** + fixtures **pendientes**
- `ScenarioEngine` — enumera compleciones y prueba si una propiedad vale en **todos** los mundos

**Hint:** Los partidos pendientes se completan con marcadores canónicos `1-0` / `0-0` / `0-1` (`PlayedMatch.from_outcome`).


In [ ]:
# Mundo mínimo: A gana, empata y pierde
tiny = World(
    teams=("A", "B", "C", "D"),
    results=frozenset({
        PlayedMatch("A", "B", 2, 0),  # A gana
        PlayedMatch("A", "C", 1, 1),  # empate
        PlayedMatch("A", "D", 0, 1),  # A pierde
        PlayedMatch("B", "C", 0, 0),
        PlayedMatch("B", "D", 0, 0),
        PlayedMatch("C", "D", 0, 0),
    }),
)

print("Partidos de A:")
for m in tiny.matches_involving("A"):
    print(f"  {m.home} {m.home_goals}-{m.away_goals} {m.away}  → {m.outcome}")

print("Rivales de A:", tiny.group_rivals("A"))


## Paso 1: Implementar `points`

Abre `src/worldcup/rules.py` e implementa `points(world, team)`.

**Regla:** victoria = 3, empate = 1, derrota = 0.

**Hint:** Usa `world.matches_involving(team)` y `match.outcome`. Compara `team` con `match.home` / `match.away` para saber si la victoria es propia.


In [ ]:
from src.worldcup.rules import points

# Tras implementar: A = 3 + 1 + 0 = 4
print(f"Puntos de A: {points(tiny, 'A')}")
assert points(tiny, "A") == 4
print("OK: points(A) == 4")


## Paso 2: Implementar `goal_difference`

Implementa `goal_difference(world, team)` = goles a favor − goles en contra.

**Hint:** Si `team` es local, GF = `home_goals` y GA = `away_goals`; si es visitante, al revés.


In [ ]:
from src.worldcup.rules import goal_difference

# A: 2-0, 1-1, 0-1 → GF=3, GA=2 → DG=+1
print(f"DG de A: {goal_difference(tiny, 'A')}")
assert goal_difference(tiny, "A") == 1
print("OK: goal_difference(A) == 1")


## Paso 3: Ranking — `teams_below` y `teams_above`

En este taller el orden de clasificación usa solo dos criterios, en este orden:

1. **Puntos** (más es mejor)
2. **Diferencia de goles** (más es mejor)

En Python puedes comparar tuplas: `(pts_A, gd_A) > (pts_B, gd_B)`.

- `teams_below(world, team)` — cuántos rivales quedan **estrictamente por debajo**
- `teams_above(world, team)` — cuántos rivales quedan **estrictamente por encima**

**Clasificación / eliminación (idea):**
- top-2 asegurado ⟺ al menos **2** rivales quedan debajo
- eliminado ⟺ al menos **2** rivales quedan encima

**Hint:** Usa `world.group_rivals(team)` y reutiliza `points` / `goal_difference`.


In [ ]:
from src.worldcup.rules import teams_above, teams_below

# Líder claro: A gana todo
leader = World(
    teams=("A", "B", "C", "D"),
    results=frozenset({
        PlayedMatch("A", "B", 1, 0),
        PlayedMatch("A", "C", 2, 0),
        PlayedMatch("A", "D", 3, 0),
        PlayedMatch("B", "C", 0, 0),
        PlayedMatch("B", "D", 0, 0),
        PlayedMatch("C", "D", 0, 0),
    }),
)

print(f"teams_below(A) = {teams_below(leader, 'A')}  (esperado: 3)")
print(f"teams_above(D) = {teams_above(leader, 'D')}  (esperado: 3)")
assert teams_below(leader, "A") == 3
assert teams_above(leader, "D") == 3
print("OK: líder / último")


### Empate en puntos: decide la DG

Cuando dos equipos tienen los mismos puntos, **solo la diferencia de goles** rompe el empate en este modelo (no hay H2H ni goles a favor como criterio extra).

En el mundo de la siguiente celda, A y B empatan a 5 puntos; A tiene mejor DG.


In [ ]:
# A5(+2), B5(+1), C1, D1 — misma cantidad de pts; DG decide
tied = World(
    teams=("A", "B", "C", "D"),
    results=frozenset({
        PlayedMatch("A", "C", 2, 0),
        PlayedMatch("B", "D", 1, 0),
        PlayedMatch("A", "D", 0, 0),
        PlayedMatch("B", "C", 0, 0),
        PlayedMatch("A", "B", 0, 0),
        PlayedMatch("C", "D", 0, 0),
    }),
)

print(f"A: pts={points(tied, 'A')} DG={goal_difference(tied, 'A')} below={teams_below(tied, 'A')}")
print(f"B: pts={points(tied, 'B')} DG={goal_difference(tied, 'B')} below={teams_below(tied, 'B')}")
assert points(tied, "A") == points(tied, "B") == 5
assert goal_difference(tied, "A") > goal_difference(tied, "B")
assert teams_below(tied, "A") > teams_below(tied, "B")
print("OK: la DG rompe el empate a favor de A")


## Paso 4: `PartialState` y escenarios (`3^k`)

Un `PartialState` tiene:

- `known` — marcadores ya fijados
- `pending` — fixtures sin jugar
- `teams` — los 4 del grupo

`ScenarioEngine.scenarios(state)` genera **todas** las compleciones: cada pendiente admite 3 resultados (1-0 / 0-0 / 0-1), así que hay **`3^k` mundos** si hay `k` pendientes.

In [ ]:
engine = ScenarioEngine()

# 2 pendientes → 3^2 = 9 mundos
state_demo = PartialState(
    teams=("A", "B", "C", "D"),
    known=frozenset({
        PlayedMatch("A", "B", 1, 0),
        PlayedMatch("C", "D", 0, 0),
        PlayedMatch("A", "C", 0, 0),
        PlayedMatch("B", "D", 0, 0),
    }),
    pending=(Fixture("A", "D"), Fixture("B", "C")),
)

worlds = list(engine.scenarios(state_demo))
print(f"Pendientes: {len(state_demo.pending)}")
print(f"Mundos generados: {len(worlds)}  (esperado: 3**{len(state_demo.pending)} = {3 ** len(state_demo.pending)})")
assert len(worlds) == 9

# Un mundo: mira cómo se rellenan los pendientes con marcadores canónicos
w0 = worlds[0]
print("Ejemplo de compleción (pendientes):")
for fix in state_demo.pending:
    m = w0.played(fix.home, fix.away)
    print(f"  {m.home} {m.home_goals}-{m.away_goals} {m.away} ({m.outcome.value})")


## Paso 5: `holds_in_all` y clasificación matemática

`engine.holds_in_all(pred, state)` es verdadero **si y solo si** `pred(world)` vale en **cada** escenario.

Eso modela entailment: *a partir de lo conocido, la conclusión es inevitable* (KB ⊨ propiedad).

**Hint:** Usa `engine.holds_in_all(lambda w: ..., state)`.

**Hecho base vs. conclusión:** los partidos en `state.known` son datos; `definitely_qualified` / `definitely_eliminated` son propiedades derivadas que solo se afirman si valen ∀ mundos.


In [ ]:
from src.worldcup.rules import definitely_qualified

# Caso estilo México (Grupo A): MEX ya no puede salir del top-2
teams = ("MEX", "RSA", "KOR", "CZE")
known = [
    PlayedMatch("MEX", "RSA", 2, 0),
    PlayedMatch("KOR", "CZE", 1, 1),
    PlayedMatch("MEX", "KOR", 1, 0),
    PlayedMatch("RSA", "CZE", 1, 0),
]
pending = [Fixture("MEX", "CZE"), Fixture("RSA", "KOR")]
mex_state = PartialState(teams=teams, known=frozenset(known), pending=tuple(pending))

print(f"Mundos: {3 ** len(mex_state.pending)}")
print(f"MEX clasificado matemáticamente: {definitely_qualified(engine, mex_state, 'MEX')}")
print(f"RSA clasificado matemáticamente: {definitely_qualified(engine, mex_state, 'RSA')}")
assert definitely_qualified(engine, mex_state, "MEX") is True
assert definitely_qualified(engine, mex_state, "RSA") is False
print("OK: clinch de MEX")


## Paso 6: Implementar `definitely_eliminated`

Misma idea que el paso anterior, pero con `teams_above(...) >= 2` en **todos** los escenarios.

Si un equipo aún puede acabar 1.º o 2.º en algún mundo, **no** está eliminado (aunque vaya mal en la tabla actual).


In [ ]:
from src.worldcup.rules import definitely_eliminated

# D perdió los 3 cruzados ya jugados; solo queda B-C pendiente
elim_state = PartialState(
    teams=("A", "B", "C", "D"),
    known=frozenset({
        PlayedMatch("A", "D", 1, 0),
        PlayedMatch("B", "D", 1, 0),
        PlayedMatch("C", "D", 1, 0),
        PlayedMatch("A", "B", 0, 0),
        PlayedMatch("A", "C", 0, 0),
    }),
    pending=(Fixture("B", "C"),),
)

print(f"D eliminado: {definitely_eliminated(engine, elim_state, 'D')}")
print(f"D clasificado: {definitely_qualified(engine, elim_state, 'D')}")
assert definitely_eliminated(engine, elim_state, "D") is True
assert definitely_qualified(engine, elim_state, "D") is False
print("OK: D está eliminado con partidos pendientes")


### Contraejemplo: cuando *no* se puede afirmar

`engine.counterexample(pred, state)` devuelve un mundo donde `pred` falla (o `None` si vale en todos).

Útil para ver *por qué* RSA aún no clasifica: basta un escenario donde no tiene ≥2 rivales debajo.


In [ ]:
# ¿RSA tiene siempre ≥2 debajo? Buscamos un contraejemplo
pred_rsa = lambda w: teams_below(w, "RSA") >= 2
cex = engine.counterexample(pred_rsa, mex_state)
print("Contraejemplo para RSA top-2:", cex is not None)
if cex is not None:
    print(f"  pts RSA={points(cex, 'RSA')} below={teams_below(cex, 'RSA')}")
    print(f"  pts MEX={points(cex, 'MEX')} KOR={points(cex, 'KOR')} CZE={points(cex, 'CZE')}")


## Limitaciones del modelo

1. **Mejores terceros** (top-8 entre los 3.º de grupo): **no** entran en la lógica de escenarios. La UI los calcula en Python *después* del último partido (`best_third_teams` en `matches.py`).
2. **Marcadores canónicos** en pendientes: solo se enumeran `1-0` / `0-0` / `0-1`. Eso basta para explorar victorias/empates/derrotas (y una DG simbólica), pero no cubre todos los marcadores reales posibles.
3. **Dominio CSV:** `load_matches` + `state_for(schedule, played, group)` construyen el `PartialState` de un grupo a partir del calendario.

La siguiente celda ilustra el dominio:


In [ ]:
# Dominio: cargar CSV y construir estado de un grupo (explorar, no implementar)
schedule = load_matches()
print(f"Partidos en CSV: {len(schedule)}")

# Ejemplo: tras 4 partidos del grupo A (jornada 1+2), quedan 2 pendientes → 9 mundos
group_a = [m for m in schedule if m.group == "a"]
played_a = group_a[:4] 
st = state_for(schedule, played_a, "a")
print(f"Equipos: {st.teams}")
print(f"Conocidos: {len(st.known)} | Pendientes: {len(st.pending)}")
print(f"Escenarios: {3 ** len(st.pending)}")
for fix in st.pending:
    print(f"  pendiente: {fix.home} vs {fix.away}")


## Verificar tu implementación

Ejecuta los tests unitarios con este comando:

```bash
uv run pytest tests/test_predicates.py -v
```

O ejecuta la celda siguiente.


In [ ]:
# Verificar implementación de predicados
!cd .. && uv run pytest tests/test_predicates.py -v